In [ ]:
# Tank 8: 목표 수위 50%를 1시간 제어 후 두 밸브 닫기
# 실행: Shift+Enter / 중간 중지: Jupyter의 ■ 버튼
# 강제 소비가 계속되면 제어 종료 후 수위는 다시 내려갑니다.
import time
from pymodbus.client import ModbusTcpClient

HOST = '192.168.0.226'
PORT = 502
DEVICE_ID = 1
LEVEL_ADDRESS = 16
FILL_ADDRESS = 16
DRAIN_ADDRESS = 17
TARGET = 500  # Scale 100: 수위 0~1000 중 500 = 50%
MAX_SECONDS = 3600

def check(response):
    if response.isError():
        raise RuntimeError(f'Modbus 오류: {response}')
    return response

def hold_tank8_at_50():
    tank_client = ModbusTcpClient(HOST, port=PORT, timeout=2, retries=0)
    if not tank_client.connect():
        tank_client.close()
        raise ConnectionError(f'Factory I/O 연결 실패: {HOST}:{PORT}')

    from datetime import datetime, timedelta, timezone
    end_time = datetime.now(timezone(timedelta(hours=9))) + timedelta(seconds=MAX_SECONDS)
    print(f'1시간 제어 시작 | 종료 예정(한국시간): {end_time:%Y-%m-%d %H:%M:%S}', flush=True)
    integral = 500.0
    previous_time = time.monotonic()
    deadline = previous_time + MAX_SECONDS
    try:
        check(tank_client.write_register(DRAIN_ADDRESS, 0, device_id=DEVICE_ID))
        while time.monotonic() < deadline:
            now = time.monotonic()
            dt = min(now - previous_time, 1.0)
            previous_time = now
            level = check(tank_client.read_input_registers(LEVEL_ADDRESS, count=1, device_id=DEVICE_ID)).registers[0]
            if not 0 <= level <= 1000:
                raise ValueError(f'수위값 {level}: Scale 100 및 매핑을 확인하세요.')

            error = TARGET - level
            # PI 제어: 현재 오차와 누적 오차를 이용해 급수량 계산
            candidate = integral + 1.5 * error * dt
            raw = 4 * error + candidate
            if 0 <= raw <= 1000 or (raw > 1000 and error < 0) or (raw < 0 and error > 0):
                integral = max(0.0, min(1000.0, candidate))
            command = round(max(0, min(1000, 4 * error + integral)))
            check(tank_client.write_register(FILL_ADDRESS, command, device_id=DEVICE_ID))
            print(f'\rTank 8 수위: {level / 10:.1f}% | 목표: 50% | 급수: {command / 10:.1f}%', end='', flush=True)
            time.sleep(0.2)
        print('\n1시간 수위 제어를 완료했습니다.')
    except KeyboardInterrupt:
        print('\n사용자가 제어를 중지했습니다.')
    finally:
        try:
            check(tank_client.write_registers(FILL_ADDRESS, [0, 0], device_id=DEVICE_ID))
            valves = check(tank_client.read_holding_registers(FILL_ADDRESS, count=2, device_id=DEVICE_ID)).registers
            print(f'\n종료: 급수·배수 명령값 = {valves} (정상: [0, 0])')
        finally:
            tank_client.close()

hold_tank8_at_50()
